# MovieRec-BDA: Big Data Movie Recommendation System Using MongoDB
## BDA Lab Mini-Project Notebook Demonstration
---
This notebook demonstrates the end-to-end data processing, MongoDB aggregation analytics, and content-based recommendation vectorization pipeline.

### 1. Import Dependencies

In [ ]:
import os
import sys
import json
from pathlib import Path
import pandas as pd
import numpy as np
import plotly.express as px
from dotenv import load_dotenv

# Load environment variables
load_dotenv()
print("Libraries loaded successfully!")

### 2. Dataset Loading & Inspection

In [ ]:
data_dir = Path("../data")
movies_file = data_dir / "ml-latest-small" / "movies.csv"
ratings_file = data_dir / "ml-latest-small" / "ratings.csv"

movies_df = pd.read_csv(movies_file)
ratings_df = pd.read_csv(ratings_file)

print(f"Movies shape: {movies_df.shape}")
print(f"Ratings shape: {ratings_df.shape}")
movies_df.head()

### 3. Data Cleaning & Validation

In [ ]:
# Parse genres and drop invalid/null rows
movies_df['genre_list'] = movies_df['genres'].apply(lambda x: x.split('|') if isinstance(x, str) and x != '(no genres listed)' else [])
valid_ratings = ratings_df[(ratings_df['rating'] >= 0.5) & (ratings_df['rating'] <= 5.0)]

print(f"Valid Movies: {len(movies_df):,}")
print(f"Valid Ratings: {len(valid_ratings):,}")

### 4. Content-Based Multi-Hot Genre Vectorization & Cosine Similarity

In [ ]:
# Extract unique genres
all_genres = sorted(list(set(g for genres in movies_df['genre_list'] for g in genres)))
genre_map = {g: i for i, g in enumerate(all_genres)}

# Create Multi-Hot Matrix
matrix = np.zeros((len(movies_df), len(all_genres)))
for i, row in movies_df.iterrows():
    for g in row['genre_list']:
        if g in genre_map:
            matrix[i, genre_map[g]] = 1.0

# Normalize and compute Cosine Similarity
norms = np.maximum(np.linalg.norm(matrix, axis=1, keepdims=True), 1e-9)
norm_matrix = matrix / norms
sim_matrix = np.dot(norm_matrix, norm_matrix.T)

print(f"Multi-Hot Matrix shape: {matrix.shape}")
print(f"Similarity Matrix shape: {sim_matrix.shape}")

### 5. Recommendation Engine Function

In [ ]:
def recommend_movies(target_title, top_n=5):
    matching = movies_df[movies_df['title'] == target_title]
    if matching.empty:
        return f"Movie '{target_title}' not found."
    
    idx = matching.index[0]
    scores = sim_matrix[idx]
    
    # Rank indices
    ranked_indices = np.argsort(scores)[::-1]
    recs = []
    for i in ranked_indices:
        if i == idx:
            continue
        recs.append({
            'title': movies_df.iloc[i]['title'],
            'genres': movies_df.iloc[i]['genres'],
            'similarity_score': round(float(scores[i]), 4)
        })
        if len(recs) == top_n:
            break
    return pd.DataFrame(recs)

# Sample recommendation
recommend_movies('Toy Story (1995)', top_n=5)

### 6. Summary & Conclusion
The content-based recommendation model accurately identifies genre-matched movies using sparse vector inner products. Combined with MongoDB Atlas aggregation pipelines, the system delivers real-time analytics and scalable NoSQL database performance.